In [1]:
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize(project='gee-space-hack')

In [2]:
def get_sar(aoi, year):
    start = ee.Date.fromYMD(year, 3, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
        .median())
    ratio = s1.select('VV').divide(s1.select('VH')).rename('VV_VH_ratio')
    return s1.addBands(ratio)

In [3]:
# ======================== CONFIGURACION ========================

AOI = ee.FeatureCollection('projects/gee-space-hack/assets/SpaceHack/AOI_GreaterGuayaquil_v2')
ASSET_ROOT = 'projects/gee-space-hack/assets/SpaceHack'

# Compuestos generados en M1
COMPOSITES = {}
for year in [2022, 2023, 2024, 2025]:
    s2 = ee.Image(f'{ASSET_ROOT}/COM_{year}')
    sar = get_sar(AOI.geometry(), year)
    ndbi = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')
    COMPOSITES[year] = s2.addBands(sar).addBands(ndbi)

# Puntos de entrenamiento
TRAINING_POINTS = ee.FeatureCollection(f'{ASSET_ROOT}/training_points_v2')

# Manglares SERVIR (referencia)
MAN_2022 = ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2022')

# Bandas para clasificacion (espectrales + indices)
BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
         'NDVI', 'NDWI', 'NDMI', 'MNDWI', 'NDRE', 'SAVI', 'MMRI', 'MVI', 'EVI',
         'VV', 'VH', 'VV_VH_ratio', 'NDBI']

# Random Forest
N_TREES = 300
TRAIN_FRACTION = 0.7
SEED = 42

In [4]:
# ======================== PASO 1: PREPARAR TRAINING DATA ========================

print('=== PASO 1: Preparar datos de entrenamiento ===')

# Extraer valores de bandas del compuesto 2022 en los puntos de entrenamiento
composite_2022 = COMPOSITES[2022].select(BANDS)

# Verificar bandas disponibles
print('Bandas en compuesto:', composite_2022.bandNames().getInfo())

training_data = composite_2022.sampleRegions(
    collection=TRAINING_POINTS,
    properties=['class'],
    scale=10,
    geometries=True
)

# Filtrar puntos donde no hay datos (pixeles enmascarados por nubes)
training_data = training_data.filter(ee.Filter.notNull(BANDS))

print('Puntos con datos validos:', training_data.size().getInfo())

=== PASO 1: Preparar datos de entrenamiento ===
Bandas en compuesto: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'NDMI', 'MNDWI', 'NDRE', 'SAVI', 'MMRI', 'MVI', 'EVI', 'VV', 'VH', 'VV_VH_ratio', 'NDBI']
Puntos con datos validos: 3499


In [5]:
# ======================== PASO 2: SPLIT TRAIN/TEST ========================

print('\n=== PASO 2: Split entrenamiento/validacion ===')

training_data = training_data.randomColumn('random', SEED)
train_set = training_data.filter(ee.Filter.lt('random', TRAIN_FRACTION))
test_set = training_data.filter(ee.Filter.gte('random', TRAIN_FRACTION))

print(f'Entrenamiento (70%): {train_set.size().getInfo()}')
print(f'Validacion (30%):    {test_set.size().getInfo()}')


=== PASO 2: Split entrenamiento/validacion ===
Entrenamiento (70%): 2434
Validacion (30%):    1065


In [6]:
# ======================== PASO 3: ENTRENAR RANDOM FOREST ========================

print('\n=== PASO 3: Entrenar Random Forest ===')

classifier = ee.Classifier.smileRandomForest(**{
    'numberOfTrees': 300,
    'variablesPerSplit': None,  # por defecto sqrt(n_bands), probar con 5-7
    'minLeafPopulation': 1,     # minimo pixeles por hoja, subir a 5-10 reduce overfitting
    'bagFraction': 0.5,         # fraccion de datos por arbol, probar 0.6-0.8
    'maxNodes': None,           # None = sin limite, probar 50-100 para regularizar
    'seed': 42
}).train(
    features=train_set,
    classProperty='class',
    inputProperties=BANDS
)

# Importancia de variables
importance = ee.Dictionary(classifier.explain().get('importance'))
print('Importancia de variables:')
imp_dict = importance.getInfo()
for band, val in sorted(imp_dict.items(), key=lambda x: -x[1]):
    print(f'  {band}: {val:.2f}')


=== PASO 3: Entrenar Random Forest ===
Importancia de variables:
  VH: 824.13
  VV: 748.69
  VV_VH_ratio: 708.97
  MVI: 676.36
  B12: 668.95
  NDRE: 660.67
  B11: 659.21
  NDMI: 654.61
  MMRI: 641.48
  B2: 640.15
  B3: 639.93
  B5: 639.51
  MNDWI: 635.03
  NDWI: 632.21
  B4: 630.51
  NDBI: 626.52
  B7: 597.17
  B6: 594.43
  B8A: 588.21
  B8: 584.44
  SAVI: 584.36
  EVI: 579.52
  NDVI: 554.48


In [7]:
# ======================== PASO 4: VALIDACION ========================

print('\n=== PASO 4: Validacion ===')

validated = test_set.classify(classifier)
confusion_matrix = validated.errorMatrix('class', 'classification')

print('Matriz de confusion:')
print(confusion_matrix.getInfo())

accuracy = confusion_matrix.accuracy().getInfo()
kappa = confusion_matrix.kappa().getInfo()
print(f'Accuracy global: {accuracy:.4f}')
print(f'Kappa:           {kappa:.4f}')

# Precision y recall por clase
producers = confusion_matrix.producersAccuracy().getInfo()
consumers = confusion_matrix.consumersAccuracy().getInfo()
class_names = {1: 'Manglar', 2: 'Otra veg', 3: 'Agua', 4: 'Urbano', 5: 'Suelo'}

print('\nPor clase:')
print(f'  {"Clase":<15} {"Precision":<12} {"Recall":<12}')
for i, name in class_names.items():
    p = consumers[i][0] if i < len(consumers) else None
    r = producers[i][0] if i < len(producers) else None
    p_str = f'{p:.4f}' if isinstance(p, (int, float)) else 'N/A'
    r_str = f'{r:.4f}' if isinstance(r, (int, float)) else 'N/A'
    print(f'  {name:<15} {p_str:<12} {r_str:<12}')



=== PASO 4: Validacion ===
Matriz de confusion:
[[0, 0, 0, 0, 0, 0], [0, 220, 1, 0, 1, 0], [0, 1, 171, 0, 35, 13], [0, 0, 0, 191, 5, 11], [0, 1, 32, 20, 111, 43], [0, 1, 13, 15, 37, 143]]
Accuracy global: 0.7850
Kappa:           0.7312

Por clase:
  Clase           Precision    Recall      
  Manglar         N/A          0.9910      
  Otra veg        N/A          0.7773      
  Agua            N/A          0.9227      
  Urbano          N/A          0.5362      
  Suelo           N/A          0.6842      


In [8]:
print('\nPor clase:')
print(f'  {"Clase":<15} {"Precision":<12} {"Recall":<12}')
matrix = confusion_matrix.getInfo()
for i, name in class_names.items():
    if i < len(matrix):
        row = matrix[i]
        row_total = sum(row)
        col_total = sum(matrix[j][i] for j in range(len(matrix)))
        correct = matrix[i][i]
        precision = correct / col_total if col_total > 0 else 0
        recall = correct / row_total if row_total > 0 else 0
        print(f'  {name:<15} {precision:<12.4f} {recall:<12.4f}')


Por clase:
  Clase           Precision    Recall      
  Manglar         0.9865       0.9910      
  Otra veg        0.7880       0.7773      
  Agua            0.8451       0.9227      
  Urbano          0.5873       0.5362      
  Suelo           0.6810       0.6842      


In [9]:
# ======================== PASO 5: CLASIFICAR TODOS LOS ANOS ========================

print('\n=== PASO 5: Clasificar compuestos ===')

classified_maps = {}

for year, composite in COMPOSITES.items():
    img = composite.select(BANDS)
    classified = img.classify(classifier).rename('classification').clip(AOI.geometry())
    classified_maps[year] = classified

    # Exportar mapa clasificado
    task = ee.batch.Export.image.toAsset(**{
        'image': classified.toByte(),
        'description': f'CLASSIFIED_{year}',
        'assetId': f'{ASSET_ROOT}/CLASSIFIED_{year}',
        'scale': 10,
        'region': AOI.geometry(),
        'maxPixels': 1e10
    })
    task.start()
    print(f'Export lanzado: CLASSIFIED_{year}')



=== PASO 5: Clasificar compuestos ===
Export lanzado: CLASSIFIED_2022
Export lanzado: CLASSIFIED_2023
Export lanzado: CLASSIFIED_2024
Export lanzado: CLASSIFIED_2025


In [10]:
# ======================== PASO 6: MAPA BINARIO MANGLAR ========================

print('\n=== PASO 6: Mapas binarios manglar/no-manglar ===')

mangrove_maps = {}

for year, classified in classified_maps.items():
    # Clase 1 = manglar
    mangrove_binary = classified.eq(1).rename('manglar')
    mangrove_maps[year] = mangrove_binary

    task = ee.batch.Export.image.toAsset(**{
        'image': mangrove_binary.toByte(),
        'description': f'MANGROVE_{year}',
        'assetId': f'{ASSET_ROOT}/MANGROVE_{year}',
        'scale': 10,
        'region': AOI.geometry(),
        'maxPixels': 1e10
    })
    task.start()
    print(f'Export lanzado: MANGROVE_{year}')


=== PASO 6: Mapas binarios manglar/no-manglar ===
Export lanzado: MANGROVE_2022
Export lanzado: MANGROVE_2023
Export lanzado: MANGROVE_2024
Export lanzado: MANGROVE_2025


In [11]:
# ======================== PASO 7: CALCULO DE AREAS ========================

print('\n=== PASO 7: Areas de manglar por ano ===')

# Incluir MAN_2018 y MAN_2020 de SERVIR para serie completa
man2018_raster = (ee.Image(0)
    .paint(ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2018'), 1)
    .rename('manglar')
    .clip(AOI.geometry()))

man2020_raster = (ee.Image(0)
    .paint(ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2020'), 1)
    .rename('manglar')
    .clip(AOI.geometry()))

man2022_raster = (ee.Image(0)
    .paint(ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2022'), 1)
    .rename('manglar')
    .clip(AOI.geometry()))

# Combinar SERVIR + clasificados propios
all_mangrove_maps = {
    2018: man2018_raster,
    2020: man2020_raster,
    2022: man2022_raster,  # SERVIR (referencia)
}
all_mangrove_maps.update(mangrove_maps)  # agrega 2022(RF), 2023, 2024, 2025

# Para 2022 usar el de SERVIR (mas preciso), no el clasificado
all_mangrove_maps[2022] = man2022_raster

area_results = {}

for year in sorted(all_mangrove_maps.keys()):
    if year == 2025:
        continue
    mangrove = all_mangrove_maps[year]
    # Area en hectareas
    area_img = mangrove.multiply(ee.Image.pixelArea()).divide(10000)
    area = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=AOI.geometry(),
        scale=30,
        maxPixels=1e10
    )
    area_ha = area.get('manglar').getInfo()
    area_results[year] = area_ha
    print(f'  {year}: {area_ha:,.1f} ha')

# Tasas de cambio
print('\nTasas de cambio:')
years_sorted = sorted(area_results.keys())
for i in range(1, len(years_sorted)):
    y0, y1 = years_sorted[i-1], years_sorted[i]
    a0, a1 = area_results[y0], area_results[y1]
    diff = a1 - a0
    pct = (diff / a0) * 100 if a0 > 0 else 0
    interval = y1 - y0
    annual_rate = pct / interval if interval > 0 else 0
    sign = '+' if diff > 0 else ''
    print(f'  {y0}-{y1}: {sign}{diff:,.1f} ha ({sign}{pct:.2f}%, {sign}{annual_rate:.2f}%/ano)')


=== PASO 7: Areas de manglar por ano ===
  2018: 72,952.2 ha
  2020: 71,459.9 ha
  2022: 71,199.0 ha
  2023: 76,270.4 ha
  2024: 71,800.6 ha

Tasas de cambio:
  2018-2020: -1,492.3 ha (-2.05%, -1.02%/ano)
  2020-2022: -260.9 ha (-0.37%, -0.18%/ano)
  2022-2023: +5,071.4 ha (+7.12%, +7.12%/ano)
  2023-2024: -4,469.9 ha (-5.86%, -5.86%/ano)


In [12]:
# Solo calcular 2025
mangrove_2025 = mangrove_maps[2025]
area_img = mangrove_2025.multiply(ee.Image.pixelArea()).divide(10000)
area = area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=AOI.geometry(),
    scale=500, # Increased scale significantly to avoid memory error
    maxPixels=1e10
)
print('2025:', area.get('manglar').getInfo(), 'ha')

2025: 59514.719550251204 ha


In [13]:
# ======================== PASO 8: DETECCION DE CAMBIOS ========================

print('\n=== PASO 8: Deteccion de cambios ===')

def change_detection(map_before, map_after, year_before, year_after):
    """
    Genera mapa de cambios:
    0 = no manglar persistente
    1 = manglar persistente
    2 = ganancia (0 -> 1)
    3 = perdida (1 -> 0)
    """
    change = (map_before.multiply(2).add(map_after)).rename('change')
    # Recodificar: before*2 + after
    # 0*2+0=0 (no manglar), 0*2+1=1 (ganancia), 1*2+0=2 (perdida), 1*2+1=3 (persistente)
    # Recodificar a esquema mas legible
    change_recoded = (ee.Image(0)
        .where(change.eq(0), 0)   # no manglar persistente
        .where(change.eq(3), 1)   # manglar persistente
        .where(change.eq(1), 2)   # ganancia
        .where(change.eq(2), 3)   # perdida
        .rename('change')
        .clip(AOI.geometry()))
    return change_recoded

# Cambio principal: SERVIR 2022 vs clasificado 2025
change_22_25 = change_detection(man2022_raster, mangrove_maps[2025], 2022, 2025)

task = ee.batch.Export.image.toAsset(**{
    'image': change_22_25.toByte(),
    'description': 'CHANGE_2022_2025',
    'assetId': f'{ASSET_ROOT}/CHANGE_2022_2025',
    'scale': 10,
    'region': AOI.geometry(),
    'maxPixels': 1e10
})
task.start()
print('Export lanzado: CHANGE_2022_2025')

# Cambio completo: SERVIR 2018 vs clasificado 2025
change_18_25 = change_detection(man2018_raster, mangrove_maps[2025], 2018, 2025)

task = ee.batch.Export.image.toAsset(**{
    'image': change_18_25.toByte(),
    'description': 'CHANGE_2018_2025',
    'assetId': f'{ASSET_ROOT}/CHANGE_2018_2025',
    'scale': 10,
    'region': AOI.geometry(),
    'maxPixels': 1e10
})
task.start()
print('Export lanzado: CHANGE_2018_2025')

# Areas de cambio 2022-2025
for label, val in [('Manglar persistente', 1), ('Ganancia', 2), ('Perdida', 3)]:
    area_img = change_22_25.eq(val).multiply(ee.Image.pixelArea()).divide(10000)
    area = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=AOI.geometry(),
        scale=500, # Increased scale to resolve memory error
        maxPixels=1e10
    ).values().get(0).getInfo()
    print(f'  {label} (2022-2025): {area:,.1f} ha')



=== PASO 8: Deteccion de cambios ===
Export lanzado: CHANGE_2022_2025
Export lanzado: CHANGE_2018_2025
  Manglar persistente (2022-2025): 53,962.1 ha
  Ganancia (2022-2025): 5,669.6 ha
  Perdida (2022-2025): 17,489.8 ha


In [14]:
# ======================== PASO 9: VISUALIZACION ========================

print('\n=== PASO 9: Visualizacion ===')

Map = geemap.Map(center=[-2.2, -79.9], zoom=11)

# Fondo S2 2025
s2_bg = COMPOSITES[2025]
Map.addLayer(s2_bg, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'S2 RGB 2025', shown=False)

# Mapas clasificados
vis_class = {'min': 1, 'max': 5, 'palette': ['00cc44', 'cccc00', '0066ff', 'ff3333', 'ff8800']}
Map.addLayer(classified_maps[2025], vis_class, 'Clasificacion 2025')
Map.addLayer(classified_maps[2024], vis_class, 'Clasificacion 2024', shown=False)
Map.addLayer(classified_maps[2023], vis_class, 'Clasificacion 2023', shown=False)

# Manglar SERVIR referencia
Map.addLayer(MAN_2022, {'color': '00ff8855'}, 'MAN_2022 SERVIR', shown=False)

# Mapa de cambios 2022-2025
vis_change = {'min': 0, 'max': 3, 'palette': ['d9d9d9', '00cc44', '0066ff', 'ff0000']}
# gris=no manglar, verde=persistente, azul=ganancia, rojo=perdida
Map.addLayer(change_22_25, vis_change, 'Cambios 2022-2025')

# Mapa de cambios 2018-2025
Map.addLayer(change_18_25, vis_change, 'Cambios 2018-2025', shown=False)

Map.addLayerControl()
Map


=== PASO 9: Visualizacion ===


Map(center=[-2.2, -79.9], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…